**Autores**

* DIANA CAROLINA LEAL MARTINEZ
* DIEGO ALEJANDRO FIGUEREDO SALAMANCA
* MARIA CAMILA REYES GIL
* SILVIA CARDENAS SANTOS



# Conexion

En el siguiente codigo se conecta a la base de datos


In [1]:
# Configuración servidor base de datos transaccional
# usuario
db_user = 'DB_202615_'
#contraseña
db_psswd = ''

db_host = '157.253.236.120'
db_port = 8080
db_destino = 'DB_202615_'      # su base de datos personal

if not db_user or not db_psswd or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'

source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/WWImportersTransactional?{jdbc_opciones}'

dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

In [2]:
import os
import sys

# En la máquina virtual ya está todo instalado
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'

if IN_COLAB or IN_CODESPACES:
    %pip install -q pyspark==4.2.0 pandas==2.3.3 numpy==2.5.3 pyarrow==25.0.1 matplotlib==3.11.2

if IN_COLAB:
    # Spark 4 requiere Java 17 o superior
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f'Entorno detectado: {ENVIRONMENT}')

from pyspark.sql import functions as f, SparkSession, types as t
from pyspark.sql.functions import udf, col, length, isnan, when, count, regexp_replace
from datetime import datetime

from pathlib import Path
import urllib.request

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()
print(f'Conector JDBC: {MYSQL_JAR}')

Entorno detectado: local
Conector JDBC: C:\Users\USUARIO\.miid4103\jars\mysql-connector-j-9.7.0.jar


In [3]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
# Necesario en Windows si Python está en C:\Program Files
import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))
# Spark 4.2 en Windows requiere HADOOP_HOME
if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    (hadoop_home / 'bin' / 'winutils.exe').touch()
    os.environ['HADOOP_HOME'] = str(hadoop_home)

spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-NB04')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')    # comportamiento de Spark 3.5
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

In [4]:
# otras librerias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, sequence, to_date, year, month, dayofmonth, date_format, dayofweek
from pyspark.sql import functions as sf
from pyspark.sql import Window
from pyspark.sql.functions import row_number, trim
import re
from pyspark.sql.functions import col, abs

## Funciones

In [5]:
def obterner_dataframe_desde_csv(_PATH, _sep):
    return spark.read.load(_PATH, format="csv", sep=_sep, inferSchema="true", header='true')

def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 10000) \
      .option('numPartitions', 4) \
      .save()

# Tabla de fechas 

## Extraer

Tal como se relato en diseño de servios se traen las bases de datos 
- FuenteAreasDeServicio_Copia_E (F1)
- FuenteTiposBeneficio_Copia_E (F2)
- FuentePlanesBeneficio_Copia_E (F3)

### Fuente de area de servicios

In [6]:
sql_fuente_areas_servicios= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.FuenteAreasDeServicio_ETL
order by
	IdAreaDeServicio_T asc) as area_servivio
"""
funte_areas_servio= obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_areas_servicios, db_user, db_psswd)
funte_areas_servio.show(10)

+------------------+--------------------+-------------+--------------------+--------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|             Condado|  Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+--------------------+--------+------------+-----+--------+-----+
|             12018|NoVa10207VA038000...|        51510|     Alexandria City|Virginia|      158309| NULL|    NULL| 2018|
|             12018|NoVa10207VA038000...|        51610|   Falls Church City|Virginia|       14309| NULL|    NULL| 2018|
|             12018|NoVa10207VA038000...|        51107|      Loudoun County|Virginia|      427592|520.0|   822.0| 2018|
|             12018|NoVa10207VA038000...|        51013|    Arlington County|Virginia|      232965| 26.0|  8960.0| 2018|
|             12018|NoVa10207VA038000...|        51059|      Fairfax County|Virginia|     1139720|396.0|  2878.0| 2018|
|             12018|NoVa10207VA038000...

### Fuente tipos de beneficios

In [7]:
sql_fuente_tipos_beneficios= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.FuenteTiposBeneficio_ETL
ORDER BY IdTipoBeneficio_T ASC) as tipos_beneficios
"""
funte_tipos_beneficios= obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_tipos_beneficios, db_user, db_psswd)
funte_tipos_beneficios.show(10)

+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|    UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|                5|Abortion For Whic...|                   |   No|                   No|                     No|                                      No|                                     No| 2017|
|                5|Abortion For Whic...|                   |   No|                  Yes|                     No|                                      No|                                    Yes| 2020|


### Fuente de planes de beneficio

In [8]:
sql_fuente_planes_beneficios= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.FuentePlanesBeneficio_ETL
ORDER BY IdTipoBeneficio_T ASC) as planes_beneficios
"""
funte_planes_beneficios= obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_planes_beneficios, db_user, db_psswd)
funte_planes_beneficios.show(10)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+
|                5|         135454040|                      238|                         27|                1|20129IL0340071-00|2021-12-31|        20129|        850|           50|          NULL|
|                5|         216714036|                      238|                         45|                3|90714MS0010005-04|2017-12-31|        90714|          0|            0|          NULL|
|                5|      

## Tranformar

1. Se selecciona la fecha para obtener los rangos
   * areas de servicio

In [9]:
funte_areas_servio.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+---------+---------+
|min_fecha|max_fecha|
+---------+---------+
|     2017|     2021|
+---------+---------+



* Tipos de beneficios

In [10]:
funte_tipos_beneficios.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+---------+---------+
|min_fecha|max_fecha|
+---------+---------+
|     2017|     2021|
+---------+---------+



In [11]:
funte_planes_beneficios.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+----------+-----------+
| min_fecha|  max_fecha|
+----------+-----------+
|2017-12-31|Dec 31,2021|
+----------+-----------+



2. Derivar Anio de la fecha de F3. Para F1 y F2 construir DATE(Anio,12,31). Usar el corte anual Anio-12-31 para ambos hechos. Registrar cualquier fecha de F3 distinta del corte para validación.

* Se identifico que hay más de un formato por lo que primero se ajusta el formato

In [12]:
regex = r"[0-2]\d{3}-(0[1-9]|1[0-2])-(0[1-9]|[1-2][0-9]|3[0-1])"
funte_planes_beneficios.select("Fecha").filter(funte_planes_beneficios["Fecha"].rlike(regex)).count()

61523

In [13]:
funte_planes_beneficios.select("Fecha").filter(~funte_planes_beneficios["Fecha"].rlike(regex)).count()

9229

> 61523 registros cumplen con el formato y 9229 no cumplen con el formato

In [14]:
regex_texto = r"^[A-Za-z]{3}\s+\d{1,2},\d{4}$"
funte_planes_beneficios.filter(funte_planes_beneficios["Fecha"].rlike(regex_texto)).count()

9229

> De los cumplen, todos tienen el formato tienen mes dia, año.

* se ajuata el formato

In [15]:
funte_planes_beneficios_limpio = funte_planes_beneficios.withColumn(
    "Fecha_Estandarizada",
    sf.when(
        sf.col("Fecha").rlike(regex), 
        sf.to_date(sf.substring(sf.col("Fecha"), 1, 10), "yyyy-MM-dd"))
    .when(
        sf.col("Fecha").rlike(regex_texto), 
        sf.to_date(sf.col("Fecha"), "MMM d,yyyy"))
    .otherwise(None))

In [16]:
funte_planes_beneficios_limpio.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+-------------------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|     Fecha|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|Fecha_Estandarizada|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+----------+-------------+-----------+-------------+--------------+-------------------+
|                5|         135454040|                      238|                         27|                1|20129IL0340071-00|2021-12-31|        20129|        850|           50|          NULL|         2021-12-31|
|                5|         216714036|                      238|                         45|                3|90714MS0010005-04|2017-12-31| 

* Elimianar la columan fecha y cambiar el nombre de fecha estandarizad con fecha

In [17]:
funte_planes_beneficios_limpio= funte_planes_beneficios_limpio.drop("Fecha").withColumnRenamed("Fecha_Estandarizada", "Fecha")
funte_planes_beneficios_limpio.show(5)

+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+-------------+-----------+-------------+--------------+----------+
|IdTipoBeneficio_T|IdAreaDeServicio_T|IdCondicionDePagoCopago_T|IdCondicionDePagoCoseguro_T|IdNivelServicio_T|         IdPlan_T|IdProveedor_T|valorCopago|valorCoseguro|cantidadLimite|     Fecha|
+-----------------+------------------+-------------------------+---------------------------+-----------------+-----------------+-------------+-----------+-------------+--------------+----------+
|                5|         135454040|                      238|                         27|                1|20129IL0340071-00|        20129|        850|           50|          NULL|2021-12-31|
|                5|         216714036|                      238|                         45|                3|90714MS0010005-04|        90714|          0|            0|          NULL|2017-12-31|
|                5|      

* Se identimica de nuevo el rango

In [18]:
funte_planes_beneficios_limpio.agg(sf.min("Fecha").alias("min_fecha"), sf.max("Fecha").alias("max_fecha")).show()

+----------+----------+
| min_fecha| max_fecha|
+----------+----------+
|2017-12-31|2021-12-31|
+----------+----------+



* Con los rangos se crea una tabla de fecha.

In [19]:
fecha_inicio = "2017-12-31"
print(fecha_inicio)
fecha_fin = "2021-12-31"
print(fecha_fin)

2017-12-31
2021-12-31


In [20]:
df_rango = spark.sql(f"""
    SELECT EXPLODE(sequence(
        CAST('{fecha_inicio}' AS DATE), 
        CAST('{fecha_fin}' AS DATE), 
        INTERVAL 1 DAY
    )) as Fecha""")

df_rango.show()

+----------+
|     Fecha|
+----------+
|2017-12-31|
|2018-01-01|
|2018-01-02|
|2018-01-03|
|2018-01-04|
|2018-01-05|
|2018-01-06|
|2018-01-07|
|2018-01-08|
|2018-01-09|
|2018-01-10|
|2018-01-11|
|2018-01-12|
|2018-01-13|
|2018-01-14|
|2018-01-15|
|2018-01-16|
|2018-01-17|
|2018-01-18|
|2018-01-19|
+----------+
only showing top 20 rows


* se extrae el Dia, Mes, el año.

In [21]:
Tabla_Fecha = df_rango.selectExpr(
    "Fecha as Fecha",
    "day(Fecha) as Dia",
    "month(Fecha) as Mes",
    "year(Fecha) as anioa"
)  
Tabla_Fecha.show(10)

+----------+---+---+-----+
|     Fecha|Dia|Mes|anioa|
+----------+---+---+-----+
|2017-12-31| 31| 12| 2017|
|2018-01-01|  1|  1| 2018|
|2018-01-02|  2|  1| 2018|
|2018-01-03|  3|  1| 2018|
|2018-01-04|  4|  1| 2018|
|2018-01-05|  5|  1| 2018|
|2018-01-06|  6|  1| 2018|
|2018-01-07|  7|  1| 2018|
|2018-01-08|  8|  1| 2018|
|2018-01-09|  9|  1| 2018|
+----------+---+---+-----+
only showing top 10 rows


3. Generar Id_Fecha como entero AAAAMMDD y Fecha como DATE, visualizada AAAA-MM-DD. Eliminar repeticiones de año.

In [22]:
Tabla_Fecha_con_id = Tabla_Fecha.withColumn("ID_Fecha_T",regexp_replace(col("Fecha"), "-", ""))
Tabla_Fecha_con_id.show()

+----------+---+---+-----+----------+
|     Fecha|Dia|Mes|anioa|ID_Fecha_T|
+----------+---+---+-----+----------+
|2017-12-31| 31| 12| 2017|  20171231|
|2018-01-01|  1|  1| 2018|  20180101|
|2018-01-02|  2|  1| 2018|  20180102|
|2018-01-03|  3|  1| 2018|  20180103|
|2018-01-04|  4|  1| 2018|  20180104|
|2018-01-05|  5|  1| 2018|  20180105|
|2018-01-06|  6|  1| 2018|  20180106|
|2018-01-07|  7|  1| 2018|  20180107|
|2018-01-08|  8|  1| 2018|  20180108|
|2018-01-09|  9|  1| 2018|  20180109|
|2018-01-10| 10|  1| 2018|  20180110|
|2018-01-11| 11|  1| 2018|  20180111|
|2018-01-12| 12|  1| 2018|  20180112|
|2018-01-13| 13|  1| 2018|  20180113|
|2018-01-14| 14|  1| 2018|  20180114|
|2018-01-15| 15|  1| 2018|  20180115|
|2018-01-16| 16|  1| 2018|  20180116|
|2018-01-17| 17|  1| 2018|  20180117|
|2018-01-18| 18|  1| 2018|  20180118|
|2018-01-19| 19|  1| 2018|  20180119|
+----------+---+---+-----+----------+
only showing top 20 rows


## Cargar

Se emplea la función de *"guardar_db"* para cargar la base de datos. Se emplea el sufijo "_ELT" para diferenciar las bases de datos.

In [23]:
guardar_db(dest_db_connection_string, Tabla_Fecha_con_id,f'{db_destino}.Fecha_ELT', db_user, db_psswd)

# Codicione de pago

## Extraer

### Fuente condiciones de pago

En este punto, aparte de importar la base, se importan los valores unicos por ID, pues se ha detectado previamente que son duplicados.
> **Regla del negocio:**  Puede que existan todavía duplicados de los registros en las fuentes, por favor revisar y eliminarlos.

In [24]:
sql_condiciones_pago= """
(SELECT DISTINCT
    IdCondicionesDePago_T, Descripcion, Tipo
FROM
    RaSaTransaccional_ETL.FuenteCondicionesDePago_ETL
ORDER BY IdCondicionesDePago_T ASC
) as condiciones_pago
"""
fuente_condiciones_pago= obtener_dataframe_de_bd(source_db_connection_string, sql_condiciones_pago, db_user, db_psswd)
fuente_condiciones_pago.show(10)

+---------------------+--------------------+-----------+
|IdCondicionesDePago_T|         Descripcion|       Tipo|
+---------------------+--------------------+-----------+
|                    9|           No Charge|Coseguridad|
|                   17|Copay per Day aft...|   Copagado|
|                   18|No Charge after d...|   Coseguro|
|                   27|Coinsurance after...|   Coseguro|
|                   34|      Not Applicable|   Copagado|
|                   36|      Not Applicable|   Coseguro|
|                   45|         Coinsurance|   Coseguro|
|                   51|           No Charge|     Copago|
|                   68|Copay per Stay af...|     Copago|
|                   85|Copay per Day bef...|     Copago|
+---------------------+--------------------+-----------+
only showing top 10 rows


## Tranformar

*  Identificar valores únicos, mapear variantes reconocibles del tipo a Copago o Coseguro. Valores no clasificables se modifican en el condicional.
  > **Regla del negocio:**
> * En la fuente de condiciones de pago se eliminaron los registros con el mismo tipo y descripción que tenía diferentes identificadores.
> * En la fuente de condiciones de pago a. Se actualizaron los registros con tipo NaN b. Sin embargo, siguen existiendo errores de consistencia, se le pide corregir el tipo a solo Copago y Coseguro.

In [25]:
fuente_condiciones_pago.columns

['IdCondicionesDePago_T', 'Descripcion', 'Tipo']

In [26]:
fuente_condiciones_pago.select('Tipo').distinct().show()

+-----------+
|       Tipo|
+-----------+
|Coseguridad|
|     Copago|
|   Copagado|
|   Coseguro|
+-----------+



> tal como idica la norma toda via hay errores.

Se acontinuación se hace un condicional

In [27]:
fuente_condiciones_pago_limpio = fuente_condiciones_pago.withColumn(
    "Tipo_Estandarizada",
    sf.when(sf.col("Tipo") == "Coseguridad", "Coseguro")
    .when(sf.col("Tipo") == "Copagado", "Copago")
    .otherwise(sf.col("Tipo")),
)

In [28]:
fuente_condiciones_pago_limpio.show()

+---------------------+--------------------+-----------+------------------+
|IdCondicionesDePago_T|         Descripcion|       Tipo|Tipo_Estandarizada|
+---------------------+--------------------+-----------+------------------+
|                    9|           No Charge|Coseguridad|          Coseguro|
|                   17|Copay per Day aft...|   Copagado|            Copago|
|                   18|No Charge after d...|   Coseguro|          Coseguro|
|                   27|Coinsurance after...|   Coseguro|          Coseguro|
|                   34|      Not Applicable|   Copagado|            Copago|
|                   36|      Not Applicable|   Coseguro|          Coseguro|
|                   45|         Coinsurance|   Coseguro|          Coseguro|
|                   51|           No Charge|     Copago|            Copago|
|                   68|Copay per Stay af...|     Copago|            Copago|
|                   85|Copay per Day bef...|     Copago|            Copago|
|           

In [29]:
fuente_condiciones_pago_limpio= fuente_condiciones_pago_limpio.drop("Tipo").withColumnRenamed("Tipo_Estandarizada", "Tipo")
fuente_condiciones_pago_limpio.show()

+---------------------+--------------------+--------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|
+---------------------+--------------------+--------+
|                    9|           No Charge|Coseguro|
|                   17|Copay per Day aft...|  Copago|
|                   18|No Charge after d...|Coseguro|
|                   27|Coinsurance after...|Coseguro|
|                   34|      Not Applicable|  Copago|
|                   36|      Not Applicable|Coseguro|
|                   45|         Coinsurance|Coseguro|
|                   51|           No Charge|  Copago|
|                   68|Copay per Stay af...|  Copago|
|                   85|Copay per Day bef...|  Copago|
|                  102|Copay per Stay be...|  Copago|
|                  119|Copay per Day wit...|  Copago|
|                  136|Copay per Stay wi...|  Copago|
|                  153|Copay after deduc...|  Copago|
|                  170|Copay before dedu...|  Copago|
|                  187|Copay

* Asignar una PK

In [30]:
fuente_condiciones_pago_limpio = fuente_condiciones_pago_limpio.coalesce(1).withColumn('IdCondicionesDePago_DWH', f.monotonically_increasing_id() + 1)
fuente_condiciones_pago_limpio.show()

+---------------------+--------------------+--------+-----------------------+
|IdCondicionesDePago_T|         Descripcion|    Tipo|IdCondicionesDePago_DWH|
+---------------------+--------------------+--------+-----------------------+
|                    9|           No Charge|Coseguro|                      1|
|                   17|Copay per Day aft...|  Copago|                      2|
|                   18|No Charge after d...|Coseguro|                      3|
|                   27|Coinsurance after...|Coseguro|                      4|
|                   34|      Not Applicable|  Copago|                      5|
|                   36|      Not Applicable|Coseguro|                      6|
|                   45|         Coinsurance|Coseguro|                      7|
|                   51|           No Charge|  Copago|                      8|
|                   68|Copay per Stay af...|  Copago|                      9|
|                   85|Copay per Day bef...|  Copago|           

## Cargar

In [31]:
guardar_db(dest_db_connection_string, fuente_condiciones_pago_limpio,f'{db_destino}.CondicionDePago_ELT', db_user, db_psswd)

# niveles de servicio


In [32]:
sql_niveles_servicio= """
(SELECT 
    *
FROM
    RaSaTransaccional_ETL.NivelesDeServicio
) as niveles_servicio
"""
fuente_niveles_servicio= obtener_dataframe_de_bd(source_db_connection_string, sql_niveles_servicio, db_user, db_psswd)
fuente_niveles_servicio.show(10)

+---------------------+-------------------+---------------+
|IdNivelDeServicio_DWH|IdNivelDeServicio_T|    Descripcion|
+---------------------+-------------------+---------------+
|                    1|                  1|        Nivel 1|
|                    2|                  2|        Nivel 2|
|                    3|                  3|Fuera de la red|
+---------------------+-------------------+---------------+



## Extraer

## Tranformar

> **Regla de negocio** En la fuente de áreas de servicio
> a. Se eliminaron los registros con diferentes IdGeografia cuyo contenido en los demás atributos era el mismo. b. Se corrigieron los registros con IdAreaDeServicio e IdGeografia nulos. c. Las fechas erróneas fueron actualizadas por las correctas. d. Se le pide corregir los registros con áreas negativas, multiplicando por -1 esos valores. e. Finalmente, los que presentan un error de digitación en la población actual con 3 ceros y un uno al final del número (0001) debe corregirlos y quitarles estos últimos 4 dígitos, de tal forma que si el valor era 27540001 debe quedar 2754.

## Cargar

In [33]:
guardar_db(dest_db_connection_string, fuente_niveles_servicio,f'{db_destino}.NivelesDe_servicio_ELT', db_user, db_psswd)

# cierre de pysapark

In [34]:
spark.stop()